# Estimation de la densité mammaire en régression ordinale
### VinDr-Mammo · BI-RADS A/B/C/D · Cross-entropy vs CORN · Quadratic Weighted Kappa

**Objectif** : montrer qu'une fonction de perte *ordinal-aware* (CORN, Shi et al. 2022) surpasse
une cross-entropy nominale pour prédire la densité mammaire BI-RADS, qui est une variable
**ordinale** (A < B < C < D) et non nominale.

**Protocole de comparaison** : backbone identique (ResNet18 pré-entraîné), optimiseur identique,
hyperparamètres identiques, même seed. **Seules** la tête de sortie et la loss changent.

**Métrique principale** : Quadratic Weighted Kappa (QWK), qui pénalise une confusion A↔D bien plus
qu'une confusion B↔C — ce qui correspond au coût clinique réel.

---
**Avant de commencer** : `Exécution > Modifier le type d'exécution > GPU (T4)`.

## 1. Environnement

In [ ]:
!nvidia-smi -L
import torch, sys
print("torch", torch.__version__, "| CUDA dispo:", torch.cuda.is_available())
print("python", sys.version.split()[0])
!pip install -q pydicom

## 2. Téléchargement d'un sous-ensemble de VinDr-Mammo

Le dataset complet fait ~50 Go. Pour ce proof of concept on télécharge un **sous-ensemble
stratifié** d'études (~3-6 Go), largement suffisant pour comparer les deux losses.

**Prérequis** : compte PhysioNet *credentialed* + DUA VinDr-Mammo signée sur
https://physionet.org/content/vindr-mammo/1.0.0/

On stratifie **au niveau étude** (et non image) : les 4 vues d'une même patiente restent
toujours du même côté du split, sinon il y a fuite de données.

In [ ]:
import getpass, os, subprocess, pandas as pd, numpy as np

PHYSIONET_USER = input("Identifiant PhysioNet : ").strip()
PHYSIONET_PASS = getpass.getpass("Mot de passe PhysioNet : ")

BASE = "https://physionet.org/files/vindr-mammo/1.0.0"
DATA_ROOT = "/content/vindr_subset"
os.makedirs(DATA_ROOT, exist_ok=True)

# Étape 1 : les CSV d'annotations seulement (quelques Mo) -> on a tous les labels
for name in ["breast-level_annotations.csv", "metadata.csv"]:
    target = os.path.join(DATA_ROOT, name)
    if not os.path.exists(target) or os.path.getsize(target) == 0:
        rc = subprocess.run(["wget", "-q", "--user", PHYSIONET_USER,
                             "--password", PHYSIONET_PASS,
                             "-O", target, f"{BASE}/{name}"]).returncode
        assert rc == 0 and os.path.getsize(target) > 0, (
            f"Echec sur {name} : verifie tes identifiants et que la DUA est signee.")
    print(f"OK {name} ({os.path.getsize(target)/1e6:.2f} Mo)")

ann = pd.read_csv(os.path.join(DATA_ROOT, "breast-level_annotations.csv"))
print("\nColonnes :", list(ann.columns))
print("\nDistribution de la densite BI-RADS (niveau image, tout le dataset) :")
print(ann["breast_density"].value_counts().sort_index())
print("\nSplit officiel :")
print(ann["split"].value_counts())
ann.head()

Note sur la distribution : VinDr-Mammo est **déséquilibré** — les catégories B et C dominent
largement, A et D sont rares. C'est le déséquilibre réel de la population dépistée. On le gère
par rééchantillonnage pondéré à l'entraînement (`WeightedRandomSampler`), pas en supprimant des
données.

In [ ]:
# Étape 2 : sélection stratifiée d'études
N_TRAIN_STUDIES = 400   # ~1600 images -> ~4 Go. Monte à 1000+ si tu as le temps/l'espace.
N_TEST_STUDIES  = 120
SEED = 42

def parse_density(v):
    s = str(v).strip().upper().replace("_", " ")
    return "ABCD".index(s[-1]) if s and s[-1] in "ABCD" else None

ann["density_label"] = ann["breast_density"].apply(parse_density)
ann = ann.dropna(subset=["density_label"]).copy()
ann["density_label"] = ann["density_label"].astype(int)

# densité d'une étude = densité modale de ses images
study_density = (ann.groupby(["split", "study_id"])["density_label"]
                    .agg(lambda s: s.mode().iloc[0]).reset_index())

selected_ids = []
for split_name, n_target in [("training", N_TRAIN_STUDIES), ("test", N_TEST_STUDIES)]:
    sub = study_density[study_density["split"] == split_name]
    frac = min(1.0, n_target / len(sub))
    picked = (sub.groupby("density_label", group_keys=False)
                 .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))), random_state=SEED))
                 .head(n_target))
    selected_ids += picked["study_id"].astype(str).tolist()
    print(f"{split_name}: {len(picked)} etudes | densites {picked['density_label'].value_counts().sort_index().to_dict()}")

print(f"\nTotal : {len(selected_ids)} etudes ~ {len(selected_ids)*4} images DICOM")
print(f"Taille estimee : ~{len(selected_ids)*4*2.5/1024:.1f} Go")

In [ ]:
# DIAGNOSTIC : tester UN seul fichier avant de lancer les centaines d'autres.
# Si cette cellule echoue ou est tres lente, inutile de lancer la suivante.
import subprocess, os, time
r0 = ann[ann["study_id"].astype(str) == str(selected_ids[0])].iloc[0]
url = f"{BASE}/images/{r0['study_id']}/{r0['image_id']}.dicom"
print("URL :", url)

t0 = time.time()
p = subprocess.run(["wget", "-nv", "--tries=2", "--timeout=60", "--auth-no-challenge",
                    "--user", PHYSIONET_USER, "--password", PHYSIONET_PASS,
                    "-O", "/content/test.dicom", url],
                   capture_output=True, text=True)
dt = time.time() - t0
size = os.path.getsize("/content/test.dicom") if os.path.exists("/content/test.dicom") else 0
print(f"code retour : {p.returncode} | {size/1e6:.1f} Mo en {dt:.1f}s "
      f"= {size/1e6/max(dt,0.01):.1f} Mo/s")
print("stderr :", p.stderr.strip()[-500:] or "(vide)")
if p.returncode != 0 or size < 100_000:
    print("\nECHEC. 401 -> mot de passe ou DUA non signee. Timeout -> reseau/limite PhysioNet.")
else:
    est = size/1e6 * len(selected_ids) * 4 / 1024
    print(f"\nOK. Volume total estime : ~{est:.1f} Go")


In [ ]:
# Étape 3 : téléchargement des images DICOM
#
# On télécharge fichier par fichier plutôt qu'avec un wget récursif : on connaît
# déjà tous les image_id grâce au CSV, donc inutile de faire parcourir à wget les
# pages d'index de PhysioNet (lent, fragile, et masque les erreurs).
# Les erreurs wget sont capturées et affichées, la progression est visible.
import concurrent.futures, subprocess, os, time, sys

IMAGES_DIR = os.path.join(DATA_ROOT, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)
MIN_BYTES = 100_000          # un DICOM de mammographie pese plusieurs Mo

sel = set(map(str, selected_ids))
targets = [(str(r.study_id), str(r.image_id))
           for r in ann.itertuples() if str(r.study_id) in sel]
print(f"{len(targets)} fichiers DICOM a recuperer\n", flush=True)

def fetch_one(t):
    sid, iid = t
    d = os.path.join(IMAGES_DIR, sid); os.makedirs(d, exist_ok=True)
    out = os.path.join(d, f"{iid}.dicom")
    if os.path.exists(out) and os.path.getsize(out) > MIN_BYTES:
        return "skip", ""
    p = subprocess.run(
        ["wget", "-nv", "--tries=3", "--timeout=90", "--auth-no-challenge",
         "--user", PHYSIONET_USER, "--password", PHYSIONET_PASS,
         "-O", out, f"{BASE}/images/{sid}/{iid}.dicom"],
        capture_output=True, text=True)
    ok = p.returncode == 0 and os.path.exists(out) and os.path.getsize(out) > MIN_BYTES
    if not ok:
        if os.path.exists(out): os.remove(out)
        msg = (p.stderr.strip().splitlines() or ["?"])[-1]
        return "fail", f"{sid}/{iid}: {msg}"
    return "ok", ""

t0, counts, errors = time.time(), {"ok":0, "skip":0, "fail":0}, []
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(fetch_one, t) for t in targets]
    for i, f in enumerate(concurrent.futures.as_completed(futures), 1):
        status, err = f.result()
        counts[status] += 1
        if err: errors.append(err)
        if i % 20 == 0 or i == len(targets):
            el = time.time() - t0
            eta = el / i * (len(targets) - i) / 60
            print(f"  {i}/{len(targets)}  ok={counts['ok']} skip={counts['skip']} "
                  f"fail={counts['fail']}  {el/60:.1f} min ecoulees, "
                  f"~{eta:.0f} min restantes", flush=True)

print(f"\nTermine en {(time.time()-t0)/60:.1f} min : {counts}")
if errors:
    print(f"\n{len(errors)} echecs, 5 premiers :")
    for e in errors[:5]: print("  ", e)
    print("Relance cette cellule pour reessayer (les fichiers deja presents sont sautes).")
!du -sh {DATA_ROOT}

In [ ]:
# CSV filtré sur les études réellement présentes sur le disque
present = set(os.listdir(IMAGES_DIR))
img_col = "image_id" if "image_id" in ann.columns else "series_id"

def file_exists(r):
    return os.path.exists(os.path.join(IMAGES_DIR, str(r["study_id"]), f"{r[img_col]}.dicom"))

ann_sub = ann[ann["study_id"].astype(str).isin(present)].copy()
ann_sub = ann_sub[ann_sub.apply(file_exists, axis=1)].reset_index(drop=True)
ann_sub.to_csv(os.path.join(DATA_ROOT, "annotations_subset.csv"), index=False)

print(f"{len(ann_sub)} images utilisables")
print(ann_sub.groupby(["split","density_label"]).size().unstack(fill_value=0))

## 3. Inspection visuelle des données

Vérification indispensable avant tout entraînement : les DICOM VinDr sont en 16 bits et souvent
en `MONOCHROME1` (échelle inversée). Si on oublie l'inversion, le modèle apprend sur des images
en négatif — l'erreur classique en mammographie.

In [ ]:
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import matplotlib.pyplot as plt
from PIL import Image

def load_dicom(path, size=512, keep_aspect=True):
    """Lit un DICOM de mammographie -> array float32 [size,size] dans [0,1].

    Ordre des operations (important) :
      1. VOI LUT / fenetrage DICOM -> restaure le contraste diagnostique.
         Sans cela l'image est plate et peu contrastee, or la densite BI-RADS
         EST une information de contraste : on degraderait la cible elle-meme.
      2. inversion si MONOCHROME1 -> tissu dense = clair
      3. normalisation min-max
      4. redimensionnement en preservant le rapport d'aspect (une mammographie
         fait ~2800x3500 ; l'ecraser en carre deforme la surface de tissu dense)
    """
    dcm = pydicom.dcmread(path)
    try:
        arr = apply_voi_lut(dcm.pixel_array, dcm).astype(np.float32)
    except Exception:
        arr = dcm.pixel_array.astype(np.float32)
    if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = arr.max() - arr
    arr = (arr - arr.min()) / max(float(arr.max() - arr.min()), 1e-6)

    img = Image.fromarray((arr*255).astype(np.uint8))
    if keep_aspect:
        w, h = img.size
        s = size / max(w, h)
        img = img.resize((max(1,int(round(w*s))), max(1,int(round(h*s)))), Image.BILINEAR)
        canvas = Image.new("L", (size, size), color=0)     # letterbox sur fond noir
        canvas.paste(img, ((size-img.size[0])//2, (size-img.size[1])//2))
        img = canvas
    else:
        img = img.resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0

CLASS_NAMES = ["A", "B", "C", "D"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for k, ax in enumerate(axes):
    rows = ann_sub[ann_sub["density_label"] == k]
    if len(rows) == 0:
        ax.axis("off"); continue
    r = rows.iloc[0]
    img = load_dicom(os.path.join(IMAGES_DIR, str(r["study_id"]), f"{r[img_col]}.dicom"))
    ax.imshow(img, cmap="gray"); ax.set_title(f"BI-RADS densite {CLASS_NAMES[k]}"); ax.axis("off")
plt.suptitle("Densite croissante : A (adipeux) -> D (extremement dense)", y=1.02)
plt.tight_layout(); plt.show()

## 4. Décodage préalable en cache NPY

Décoder les DICOM à la volée à chaque epoch est le goulot d'étranglement principal. On décode
**une seule fois** vers un cache `.npy` en 512px uint8 : l'entraînement devient limité par le GPU
et non par le disque. Gain typique : 5 à 10x.

In [ ]:
IMG_SIZE = 512
CACHE_DIR = f"/content/cache_{IMG_SIZE}"
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_one(args):
    i, r = args
    out = os.path.join(CACHE_DIR, f"{r[img_col]}.npy")
    if os.path.exists(out):
        return
    img = load_dicom(os.path.join(IMAGES_DIR, str(r["study_id"]), f"{r[img_col]}.dicom"), IMG_SIZE)
    np.save(out, (img*255).astype(np.uint8))

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(cache_one, ann_sub.iterrows()))
print(f"Cache construit en {(time.time()-t0)/60:.1f} min")
!du -sh {CACHE_DIR}

## 5. Dataset et DataLoaders

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as tv

N_CLASSES = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BreastDensityDataset(Dataset):
    def __init__(self, df, cache_dir, transform=None):
        self.df = df.reset_index(drop=True); self.cache_dir = cache_dir; self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        arr = np.load(os.path.join(self.cache_dir, f"{r[img_col]}.npy")).astype(np.float32)/255.
        x = torch.from_numpy(arr).unsqueeze(0).repeat(3,1,1)     # 1 canal -> 3 (backbone RGB)
        if self.transform is not None: x = self.transform(x)
        return x, int(r["density_label"])

norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])   # stats ImageNet
train_tf = T.Compose([T.RandomHorizontalFlip(),
                      T.RandomAffine(degrees=8, translate=(0.05,0.05), scale=(0.95,1.05)),
                      norm])
eval_tf = norm

# Split validation DANS le training, au niveau étude (pas image) -> zéro fuite
rng = np.random.default_rng(SEED)
tr_all = ann_sub[ann_sub["split"]=="training"]
studies = tr_all["study_id"].unique(); rng.shuffle(studies)
n_val = max(1, int(0.15*len(studies))); val_studies = set(studies[:n_val])

df_tr  = tr_all[~tr_all["study_id"].isin(val_studies)]
df_val = tr_all[ tr_all["study_id"].isin(val_studies)]
df_te  = ann_sub[ann_sub["split"]=="test"]
print(f"train={len(df_tr)}  val={len(df_val)}  test={len(df_te)} images")

BATCH_SIZE = 16
# Rééchantillonnage pondéré : compense le déséquilibre A/D vs B/C
labels_tr = df_tr["density_label"].to_numpy()
counts = np.bincount(labels_tr, minlength=N_CLASSES).astype(float)
w = 1.0/np.maximum(counts[labels_tr], 1)
sampler = WeightedRandomSampler(torch.DoubleTensor(w), num_samples=len(w), replacement=True)

train_loader = DataLoader(BreastDensityDataset(df_tr, CACHE_DIR, train_tf),
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(BreastDensityDataset(df_val, CACHE_DIR, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(BreastDensityDataset(df_te, CACHE_DIR, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 6. Les deux modèles

**Baseline (nominal)** — ResNet18 + tête softmax 4 classes, entraîné en cross-entropy.
Le modèle n'a aucune notion d'ordre : confondre A avec D coûte exactement autant que confondre
B avec C.

**CORN (ordinal-aware)** — même ResNet18, mais la tête produit K-1 = 3 logits modélisant les
probabilités **conditionnelles** P(Y > k | Y ≥ k). La loss n'entraîne le seuil k que sur le
sous-ensemble des exemples tels que y ≥ k. Ce conditionnement garantit la cohérence de rang
par construction, sans imposer de biais ordonnés partagés entre tous les exemples comme le fait
CORAL.

In [ ]:
def make_backbone(pretrained=True):
    net = tv.resnet18(weights=tv.ResNet18_Weights.DEFAULT if pretrained else None)
    n_feat = net.fc.in_features
    net.fc = nn.Identity()
    return net, n_feat

class BaselineClassifier(nn.Module):
    """Nominal : softmax K classes + cross-entropy."""
    def __init__(self, n_classes=N_CLASSES, pretrained=True):
        super().__init__()
        self.backbone, n_feat = make_backbone(pretrained)
        self.head = nn.Linear(n_feat, n_classes)
    def forward(self, x): return self.head(self.backbone(x))
    @staticmethod
    def predict(logits): return logits.argmax(dim=1)

class CornClassifier(nn.Module):
    """Ordinal : K-1 probabilites CONDITIONNELLES f_k = P(Y>k | Y>=k).
    Shi, Cao & Raschka (2022), arXiv:2111.08851."""
    def __init__(self, n_classes=N_CLASSES, pretrained=True):
        super().__init__()
        self.n_classes = n_classes
        self.backbone, n_feat = make_backbone(pretrained)
        self.head = nn.Linear(n_feat, n_classes-1)
    def forward(self, x): return self.head(self.backbone(x))
    @staticmethod
    def predict(logits):
        # ORDRE CRITIQUE : chainer d'abord (regle de chainage des probas
        # conditionnelles) P(Y>k) = prod_{j<=k} sigmoid(logit_j), PUIS seuiller.
        # Seuiller avant de chainer donne un decodeur non conforme au papier,
        # qui surestime le rang (ex. logits [0.5,0.5,3.0] -> 3 au lieu de 1).
        cum_probs = torch.sigmoid(logits).cumprod(dim=1)
        return (cum_probs > 0.5).sum(dim=1).long()

def ce_loss(logits, y): return F.cross_entropy(logits, y)

def corn_loss(logits, y, n_classes=N_CLASSES):
    """Somme des BCE sur les sous-ensembles conditionnels S_k = {i : y_i >= k},
    cible binaire 1[y_i > k], normalisee par le nombre total de termes."""
    total, acc = 0, 0.0
    for k in range(n_classes-1):
        m = y >= k                                   # sous-ensemble conditionnel S_k
        if m.sum() == 0: continue
        acc = acc + F.binary_cross_entropy_with_logits(
            logits[m, k], (y[m] > k).float(), reduction="sum")
        total += int(m.sum())
    return acc / max(total, 1)

# Test du decodeur sur le cas discriminant, avant tout entrainement
_t = torch.tensor([[0.5, 0.5, 3.0], [9., 9., 9.], [-9., -9., -9.]])
assert CornClassifier.predict(_t).tolist() == [1, 3, 0], CornClassifier.predict(_t).tolist()
print("Decodage CORN conforme au papier (chainage puis seuillage).")

## 7. Métriques ordinales (QWK et compagnie)

In [ ]:
def confusion_matrix(y_true, y_pred, K=N_CLASSES):
    cm = np.zeros((K, K), dtype=np.int64)
    for t, p in zip(y_true, y_pred): cm[int(t), int(p)] += 1
    return cm

def quadratic_weighted_kappa(y_true, y_pred, K=N_CLASSES):
    """QWK : penalise les erreurs proportionnellement au CARRE de la distance de rang.
    Une confusion A<->D (distance 3) coute 9x une confusion B<->C (distance 1)."""
    O = confusion_matrix(y_true, y_pred, K).astype(float)
    n = O.sum()
    if n == 0: return 0.0
    W = np.array([[(i-j)**2/(K-1)**2 for j in range(K)] for i in range(K)])
    E = np.outer(O.sum(1), O.sum(0))/n
    den = (W*E).sum()
    return 1.0 - (W*O).sum()/den if den > 0 else 1.0

def evaluate_metrics(y_true, y_pred, K=N_CLASSES):
    d = np.abs(y_true.astype(int) - y_pred.astype(int))
    n_err = int((d > 0).sum())
    return {
        "accuracy": float((y_true == y_pred).mean()),
        "qwk": float(quadratic_weighted_kappa(y_true, y_pred, K)),
        "ordinal_mae": float(d.mean()),
        "confusion_matrix": confusion_matrix(y_true, y_pred, K).tolist(),
        "n_errors": n_err,
        "frac_err_adjacent": float((d == 1).sum()/max(n_err,1)),   # ex. B<->C
        "frac_err_distant":  float((d >= 2).sum()/max(n_err,1)),   # ex. A<->C, A<->D
    }

# Sanity check : accord parfait -> 1, inversion totale -> -1
_y = np.array([0,1,2,3])
assert abs(quadratic_weighted_kappa(_y,_y) - 1.0) < 1e-9
assert abs(quadratic_weighted_kappa(_y,_y[::-1]) + 1.0) < 1e-9
print("QWK valide.")

## 8. Boucle d'entraînement (identique pour les deux modèles)

Point méthodologique : le meilleur checkpoint est sélectionné sur le **QWK de validation**,
pas sur l'accuracy. Sélectionner sur l'accuracy reviendrait à optimiser une métrique qui ignore
l'ordre, ce qui biaiserait la comparaison en faveur de la baseline.

In [ ]:
import json, time

EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def run_eval(model, loader, predict_fn):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        logits = model(x.to(device, non_blocking=True))
        ps.append(predict_fn(logits.float().cpu()).numpy()); ys.append(y.numpy())
    return evaluate_metrics(np.concatenate(ys), np.concatenate(ps))

def train_model(kind, epochs=EPOCHS, seed=SEED):
    set_seed(seed)
    if kind == "baseline":
        model, loss_fn, predict_fn = BaselineClassifier().to(device), ce_loss, BaselineClassifier.predict
    else:
        model, loss_fn, predict_fn = CornClassifier().to(device), corn_loss, CornClassifier.predict

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_qwk, best_state, history = -1e9, None, []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot, nb = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); nb += 1
        sched.step()
        m = run_eval(model, val_loader, predict_fn)
        history.append({"epoch":ep, "train_loss":tot/nb, **{f"val_{k}":v for k,v in m.items()
                                                            if k!="confusion_matrix"}})
        print(f"[{kind} ep {ep:2d}] loss={tot/nb:.4f} val_acc={m['accuracy']:.4f} "
              f"val_QWK={m['qwk']:.4f} val_MAE={m['ordinal_mae']:.4f} ({time.time()-t0:.0f}s)")
        if m["qwk"] > best_qwk:                      # selection sur le QWK
            best_qwk = m["qwk"]
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_m = run_eval(model, test_loader, predict_fn)
    print(f"\n== TEST {kind} == acc={test_m['accuracy']:.4f} QWK={test_m['qwk']:.4f} "
          f"MAE={test_m['ordinal_mae']:.4f}")
    return {"kind":kind, "history":history, "best_val_qwk":best_qwk, "test_metrics":test_m,
            "model":model}

In [ ]:
res_baseline = train_model("baseline")

In [ ]:
res_corn = train_model("corn")

## 9. Comparaison et analyse de la structure ordinale

In [ ]:
import matplotlib.pyplot as plt

rows = []
for r in (res_baseline, res_corn):
    m = r["test_metrics"]
    rows.append({"modele": "Cross-entropy (nominal)" if r["kind"]=="baseline" else "CORN (ordinal)",
                 "accuracy": m["accuracy"], "QWK": m["qwk"], "ordinal MAE": m["ordinal_mae"],
                 "err. adjacentes": m["frac_err_adjacent"], "err. lointaines": m["frac_err_distant"]})
comp = pd.DataFrame(rows).set_index("modele")
display(comp.round(4))

dq = res_corn["test_metrics"]["qwk"] - res_baseline["test_metrics"]["qwk"]
dm = res_corn["test_metrics"]["ordinal_mae"] - res_baseline["test_metrics"]["ordinal_mae"]
dd = res_corn["test_metrics"]["frac_err_distant"] - res_baseline["test_metrics"]["frac_err_distant"]
print(f"\nDelta CORN - baseline :")
print(f"  QWK                       : {dq:+.4f}  (positif = CORN meilleur)")
print(f"  Ordinal MAE               : {dm:+.4f}  (negatif = CORN meilleur)")
print(f"  Part d'erreurs lointaines : {dd:+.1%}  (negatif = CORN se trompe 'moins loin')")

In [ ]:
# Matrices de confusion : la preuve visuelle de la structure ordinale apprise
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, r in zip(axes, (res_baseline, res_corn)):
    cm = np.array(r["test_metrics"]["confusion_matrix"])
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cm[i,j]}\n{cmn[i,j]:.0%}", ha="center", va="center",
                    fontsize=9, color="white" if cmn[i,j] > 0.5 else "black")
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES); ax.set_yticks(range(N_CLASSES), CLASS_NAMES)
    ax.set_xlabel("Predit"); ax.set_ylabel("Verite (BI-RADS densite)")
    name = "Cross-entropy (nominal)" if r["kind"]=="baseline" else "CORN (ordinal)"
    ax.set_title(f"{name}\nQWK={r['test_metrics']['qwk']:.3f}  acc={r['test_metrics']['accuracy']:.3f}")
fig.colorbar(im, ax=axes, fraction=0.025)
plt.suptitle("Un modele ordinal concentre ses erreurs pres de la diagonale", y=1.02)
plt.show()

In [ ]:
# Distribution des erreurs par distance de rang : le test direct de l'hypothese ordinale
fig, ax = plt.subplots(figsize=(8, 4.5))
width = 0.35
dists = np.arange(1, N_CLASSES)
for off, r, color in [(-width/2, res_baseline, "#c44"), (width/2, res_corn, "#2a7")]:
    cm = np.array(r["test_metrics"]["confusion_matrix"])
    n_err = cm.sum() - np.trace(cm)
    fracs = []
    for d in dists:
        cnt = sum(cm[i, j] for i in range(N_CLASSES) for j in range(N_CLASSES)
                  if abs(i-j) == d)
        fracs.append(cnt/max(n_err,1))
    label = "Cross-entropy (nominal)" if r["kind"]=="baseline" else "CORN (ordinal)"
    ax.bar(dists+off, fracs, width, label=label, color=color)
ax.set_xticks(dists, [f"{d}\n({'adjacent, ex. B/C' if d==1 else 'lointain'})" for d in dists])
ax.set_xlabel("Distance de rang de l'erreur |verite - prediction|")
ax.set_ylabel("Part des erreurs")
ax.set_title("Ou se situent les erreurs ? Plus c'est concentre a gauche, mieux la\nstructure ordinale est apprise")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Courbes d'apprentissage (QWK de validation)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for r, color in [(res_baseline, "#c44"), (res_corn, "#2a7")]:
    h = pd.DataFrame(r["history"])
    label = "Cross-entropy" if r["kind"]=="baseline" else "CORN"
    axes[0].plot(h["epoch"], h["val_qwk"], marker="o", ms=3, color=color, label=label)
    axes[1].plot(h["epoch"], h["val_ordinal_mae"], marker="o", ms=3, color=color, label=label)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("QWK (validation)"); axes[0].set_title("QWK de validation"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MAE ordinal (validation)"); axes[1].set_title("Erreur moyenne en rangs"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### Un seul seed ne suffit pas

Un écart de QWK de l'ordre de 0,01 sur un run unique n'est pas interprétable : sur données
synthétiques, le prototype montre un delta de +0,0123 ± 0,0118 selon le seed, avec un seed sur
cinq où CORN perd. La cellule suivante répète les deux entraînements sur plusieurs seeds et
rapporte les **deltas appariés** (même données, même split, seules les initialisations
changent). À lancer si le temps GPU le permet — comptez le temps d'un run × 2 × n_seeds.

In [ ]:
N_SEEDS = 3          # 3 minimum ; 5 c'est mieux
multi = {"baseline": [], "corn": []}

for s in range(N_SEEDS):
    print(f"\n{'='*30} seed {s} {'='*30}")
    for kind in ("baseline", "corn"):
        r = train_model(kind, seed=SEED + s)
        multi[kind].append(r["test_metrics"])

qwk_b = np.array([m["qwk"] for m in multi["baseline"]])
qwk_c = np.array([m["qwk"] for m in multi["corn"]])
delta = qwk_c - qwk_b            # comparaison appariee seed par seed
print(f"\nQWK baseline : {qwk_b.mean():.4f} +/- {qwk_b.std():.4f}")
print(f"QWK CORN     : {qwk_c.mean():.4f} +/- {qwk_c.std():.4f}")
print(f"Delta apparie: {delta.mean():+.4f} +/- {delta.std():.4f}")
print(f"Deltas       : {', '.join(f'{d:+.4f}' for d in delta)}")
print(f"CORN gagne sur {int((delta > 0).sum())}/{N_SEEDS} seeds")


## 10. Sauvegarde des résultats

À faire avant la fin de session : `/content` est effacé quand la session Colab se termine.

In [ ]:
OUT = "/content/results"; os.makedirs(OUT, exist_ok=True)
payload = {
    "config": {"n_train_studies": N_TRAIN_STUDIES, "n_test_studies": N_TEST_STUDIES,
               "img_size": IMG_SIZE, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
               "seed": SEED, "backbone": "resnet18"},
    "n_images": {"train": len(df_tr), "val": len(df_val), "test": len(df_te)},
    "baseline": {k: v for k, v in res_baseline.items() if k != "model"},
    "corn":     {k: v for k, v in res_corn.items()     if k != "model"},
}
with open(f"{OUT}/results.json", "w") as f: json.dump(payload, f, indent=2)
torch.save(res_baseline["model"].state_dict(), f"{OUT}/baseline_resnet18.pt")
torch.save(res_corn["model"].state_dict(),     f"{OUT}/corn_resnet18.pt")
print("Sauvegarde dans", OUT); print(os.listdir(OUT))

# Copie vers Google Drive pour ne rien perdre (decommenter)
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r {OUT} "/content/drive/MyDrive/breast_density_ordinal_results"

## 11. Pistes de suite pour la thèse

Ce notebook est un proof of concept. Les extensions naturelles :

**Comparer plus de losses ordinales** — CORAL (contrainte plus forte, biais ordonnés partagés),
la régression cumulative à seuils appris (modèle à odds proportionnels), et une simple régression
L2 sur le rang suivie d'un seuillage. CORN n'est pas systématiquement le meilleur selon la taille
de l'échantillon et le déséquilibre.

**Agréger au niveau étude** — cliniquement, la densité est rapportée par sein voire par examen,
pas par vue. Fusionner les prédictions CC et MLO (moyenne des probabilités conditionnelles, ou
attention multi-vues) devrait améliorer le QWK.

**Quantifier l'incertitude** — la densité BI-RADS a un accord inter-lecteur imparfait, surtout
aux frontières B/C. Une distribution prédictive calibrée vaut mieux qu'un point unique, et c'est
directement lié à l'axe "ordinal-aware learning objectives" du sujet de thèse.

**Tester la généralisation** — entraîner sur VinDr-Mammo et évaluer sur RSNA Screening
Mammography ou INbreast : le décalage de domaine entre fabricants d'appareils est le vrai obstacle
au déploiement clinique.

**Intervalles de confiance** — répéter sur plusieurs seeds et rapporter moyenne ± écart-type,
voire un bootstrap sur le jeu de test. Un écart de QWK de 0,01 sur un seul run ne veut rien dire.

---

**Référence CORN** : Shi, X., Cao, W., & Raschka, S. (2022). *Deep Neural Networks for
Rank-Consistent Ordinal Regression Based on Conditional Probabilities.* ICPR 2022 /
arXiv:2111.08851.

**Référence CORAL** (l'approche concurrente, à ne pas confondre) : Cao, W., Mirjalili, V., &
Raschka, S. (2020). *Rank consistent ordinal regression for neural networks with application to
age estimation.* Pattern Recognition Letters, 140, 325-331.

**Référence dataset** : Pham, H. H., Nguyen Trung, H., & Nguyen, H. Q. (2022). *VinDr-Mammo:
A large-scale benchmark dataset for computer-aided detection and diagnosis in full-field digital
mammography* (version 1.0.0). PhysioNet. https://doi.org/10.13026/br2v-7517